# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 05 — Modeling, Evaluation & SHAP

**What this notebook does:**
1. Loads `final_dataset.csv` and creates train/test split (2019–2024 train | 2025 test)
2. Builds baseline models (mean predictor, linear regression, ridge)
3. Trains main models: Random Forest, XGBoost, Gradient Boosting (default + tuned)
4. Tunes hyperparameters with RandomizedSearchCV (30 iter, 5-fold CV)
5. Evaluates with MAE, RMSE, R² on held-out 2025 test set
6. Ablation study: with vs without cluster features
7. SHAP values for model interpretability
8. Saves all trained models to `models/`

**Input:**  `data/processed/final_dataset.csv`  
**Output:** `models/`, `figures/`

In [ ]:
%pip install xgboost shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.dummy           import DummyRegressor
import xgboost as xgb
import shap

RANDOM_STATE = 42
print('✅ Imports ready.')
print(f'   XGBoost: {xgb.__version__}  |  SHAP: {shap.__version__}')

## Step 1 — Configure Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'figures')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

FINAL_PATH = os.path.join(PROC_DIR, 'final_dataset.csv')
print(f"  {'✅' if os.path.exists(FINAL_PATH) else '❌ MISSING'} final_dataset.csv")
if not os.path.exists(FINAL_PATH):
    print('     → Run 04_clustering.ipynb first!')

## Step 2 — Load Data & Apply Fixes

In [ ]:
df = pd.read_csv(FINAL_PATH)

# Fix pit lane starts (grid_position=0 → 20)
pit_lane_mask = df['grid_position'] == 0
df.loc[pit_lane_mask, 'grid_position'] = 20
df.loc[pit_lane_mask, 'position_gain'] = 20 - df.loc[pit_lane_mask, 'finish_position']
print(f'Pit lane starts fixed: {pit_lane_mask.sum()}')

# Ensure cluster columns are integers
df['driver_cluster']  = df['driver_cluster'].fillna(0).astype(int)
df['circuit_cluster'] = df['circuit_cluster'].fillna(0).astype(int)

print(f'\n✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'   Train: {(df["split"]=="train").sum():,} rows')
print(f'   Test:  {(df["split"]=="test").sum():,} rows')
print(f'   Seasons: {sorted(df["season"].unique().tolist())}')

## Step 3 — Define Features & Target

In [ ]:
FEATURE_COLS = [
    # Pace features
    'avg_lap_time_norm', 'lap_time_std', 'best_lap_time_norm',
    'avg_sector1_norm', 'avg_sector2_norm', 'avg_sector3_norm',
    # Strategy features
    'tyre_degradation_slope', 'avg_stint_length', 'number_of_pit_stops',
    # Cluster labels (key contribution of this paper)
    'driver_cluster', 'circuit_cluster',
    # Car quality control + starting position
    'teammate_pace_delta', 'grid_position',
]

TARGET_COL = 'position_gain'

missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'❌ Missing features: {missing}')
else:
    print(f'✅ All {len(FEATURE_COLS)} features present')

## Step 4 — Train / Test Split

In [ ]:
train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()

X_train = train_df[FEATURE_COLS].fillna(0)
y_train = train_df[TARGET_COL]

X_test  = test_df[FEATURE_COLS].fillna(0)
y_test  = test_df[TARGET_COL]

print(f'✅ X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'✅ X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'\nTarget distribution:')
print(f'   Train mean: {y_train.mean():.3f} | std: {y_train.std():.3f}')
print(f'   Test mean:  {y_test.mean():.3f}  | std: {y_test.std():.3f}')

## Step 5 — Evaluation Helper

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, store=True):
    train_pred = model.predict(X_tr)
    test_pred  = model.predict(X_te)
    metrics = {
        'train_mae':  mean_absolute_error(y_tr, train_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_tr, train_pred)),
        'train_r2':   r2_score(y_tr, train_pred),
        'test_mae':   mean_absolute_error(y_te, test_pred),
        'test_rmse':  np.sqrt(mean_squared_error(y_te, test_pred)),
        'test_r2':    r2_score(y_te, test_pred),
        'cv_mae': None, 'cv_std': None,
    }
    print(f'\n── {name} ──')
    print(f'  Train → MAE: {metrics["train_mae"]:.3f}  RMSE: {metrics["train_rmse"]:.3f}  R²: {metrics["train_r2"]:.3f}')
    print(f'  Test  → MAE: {metrics["test_mae"]:.3f}  RMSE: {metrics["test_rmse"]:.3f}  R²: {metrics["test_r2"]:.3f}')
    if store:
        results[name] = metrics
    return metrics, test_pred

print('✅ Evaluation helper ready.')

## Step 6 — Baseline Models

In [ ]:
mean_model = DummyRegressor(strategy='mean')
mean_model.fit(X_train, y_train)
evaluate('Baseline: Mean Predictor', mean_model, X_train, y_train, X_test, y_test)

scaler_lr = StandardScaler()
X_train_scaled = scaler_lr.fit_transform(X_train)
X_test_scaled  = scaler_lr.transform(X_test)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
evaluate('Baseline: Linear Regression', lr_model, X_train_scaled, y_train, X_test_scaled, y_test)

ridge_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
ridge_model.fit(X_train_scaled, y_train)
evaluate('Baseline: Ridge Regression', ridge_model, X_train_scaled, y_train, X_test_scaled, y_test)

for name, model, X_tr in [
    ('Baseline: Linear Regression', lr_model,    X_train_scaled),
    ('Baseline: Ridge Regression',  ridge_model, X_train_scaled),
]:
    cv_s = cross_val_score(model, X_tr, y_train, cv=5, scoring='neg_mean_absolute_error')
    results[name]['cv_mae'] = -cv_s.mean()
    results[name]['cv_std'] =  cv_s.std()
    print(f'📊 {name} — CV MAE: {-cv_s.mean():.3f} ± {cv_s.std():.3f}')

## Step 7 — Random Forest

In [ ]:
rf_default = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_default.fit(X_train, y_train)
evaluate('Random Forest (default)', rf_default, X_train, y_train, X_test, y_test)

In [ ]:
print('🔄 Tuning Random Forest...')
rf_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    rf_param_grid, n_iter=30, cv=5, scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE, verbose=0, n_jobs=-1
)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
print(f'✅ Best RF params: {rf_search.best_params_}')
evaluate('Random Forest (tuned)', rf_best, X_train, y_train, X_test, y_test)

cv_scores_rf = cross_val_score(rf_best, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
results['Random Forest (tuned)']['cv_mae'] = -cv_scores_rf.mean()
results['Random Forest (tuned)']['cv_std'] =  cv_scores_rf.std()
print(f'📊 RF (tuned) — CV MAE: {-cv_scores_rf.mean():.3f} ± {cv_scores_rf.std():.3f}')

## Step 8 — XGBoost

In [ ]:
xgb_default = xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0, eval_metric='mae')
xgb_default.fit(X_train, y_train)
evaluate('XGBoost (default)', xgb_default, X_train, y_train, X_test, y_test)

In [ ]:
print('🔄 Tuning XGBoost...')
xgb_param_grid = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [3, 4, 5, 6],
    'learning_rate':   [0.01, 0.05, 0.1, 0.2],
    'subsample':       [0.6, 0.8, 1.0],
    'colsample_bytree':[0.6, 0.8, 1.0],
    'reg_lambda':      [1, 1.5, 2],
}
xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0, eval_metric='mae'),
    xgb_param_grid, n_iter=30, cv=5, scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE, verbose=0, n_jobs=-1
)
xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_
print(f'✅ Best XGB params: {xgb_search.best_params_}')
evaluate('XGBoost (tuned)', xgb_best, X_train, y_train, X_test, y_test)

cv_scores_xgb = cross_val_score(xgb_best, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
results['XGBoost (tuned)']['cv_mae'] = -cv_scores_xgb.mean()
results['XGBoost (tuned)']['cv_std'] =  cv_scores_xgb.std()
print(f'📊 XGB (tuned) — CV MAE: {-cv_scores_xgb.mean():.3f} ± {cv_scores_xgb.std():.3f}')

## Step 9 — Gradient Boosting

In [ ]:
gb_default = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
gb_default.fit(X_train, y_train)
evaluate('Gradient Boosting (default)', gb_default, X_train, y_train, X_test, y_test)

In [ ]:
print('🔄 Tuning Gradient Boosting...')
gb_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [3, 4, 5],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.6, 0.8, 1.0],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
}
gb_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    gb_param_grid, n_iter=30, cv=5, scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE, verbose=0, n_jobs=-1
)
gb_search.fit(X_train, y_train)
gb_best = gb_search.best_estimator_
print(f'✅ Best GB params: {gb_search.best_params_}')
evaluate('Gradient Boosting (tuned)', gb_best, X_train, y_train, X_test, y_test)

cv_scores_gb = cross_val_score(gb_best, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
results['Gradient Boosting (tuned)']['cv_mae'] = -cv_scores_gb.mean()
results['Gradient Boosting (tuned)']['cv_std'] =  cv_scores_gb.std()
print(f'📊 GB (tuned) — CV MAE: {-cv_scores_gb.mean():.3f} ± {cv_scores_gb.std():.3f}')

## Step 10 — Model Comparison Table

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = 'Model'

print('=' * 80)
print('MODEL COMPARISON — 2025 held-out test set')
print('=' * 80)
print(results_df[['cv_mae', 'test_mae', 'test_rmse', 'test_r2']].to_string())

best_model_name = results_df['test_mae'].idxmin()
print(f'\n🏆 Best model: {best_model_name}')
print(f'   Test MAE: {results_df.loc[best_model_name, "test_mae"]:.4f}')
print(f'   Test R²:  {results_df.loc[best_model_name, "test_r2"]:.4f}')

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

short_names = {
    'Baseline: Mean Predictor':     'Mean\nBaseline',
    'Baseline: Linear Regression':  'Linear\nReg.',
    'Baseline: Ridge Regression':   'Ridge\nReg.',
    'Random Forest (default)':      'RF\n(default)',
    'Random Forest (tuned)':        'RF\n(tuned)',
    'XGBoost (default)':            'XGB\n(default)',
    'XGBoost (tuned)':              'XGB\n(tuned)',
    'Gradient Boosting (default)':  'GB\n(default)',
    'Gradient Boosting (tuned)':    'GB\n(tuned)',
}
colors = ['#aaaaaa','#aaaaaa','#aaaaaa','#1f77b4','#1f77b4','#e10600','#e10600','#2ca02c','#2ca02c']
names  = [short_names.get(m, m) for m in results_df.index]

for ax, metric, label in zip(axes,
    ['test_mae', 'test_rmse', 'test_r2'],
    ['Test MAE ↓', 'Test RMSE ↓', 'Test R² ↑']):
    vals = results_df[metric].values
    bars = ax.bar(names, vals, color=colors[:len(names)], edgecolor='none', alpha=0.85)
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xticklabels(names, fontsize=7, rotation=30, ha='right')
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (max(vals)-min(vals))*0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)

plt.suptitle('Model Comparison — 2025 Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/model_comparison.png')

## Step 11 — Ablation Study: Cluster Features

In [ ]:
tuned_models = {
    'Random Forest (tuned)':      rf_best,
    'XGBoost (tuned)':            xgb_best,
    'Gradient Boosting (tuned)':  gb_best,
}
best_tuned_name = min(tuned_models.keys(), key=lambda m: results[m]['test_mae'])
best_model = tuned_models[best_tuned_name]

print(f'🏆 Best tuned model: {best_tuned_name}')
print('\n🔬 Ablation study — effect of cluster features:')

FEATURES_NO_CLUSTER = [c for c in FEATURE_COLS if c not in ['driver_cluster', 'circuit_cluster']]

ablation_model = type(best_model)(**best_model.get_params())
ablation_model.fit(X_train[FEATURES_NO_CLUSTER], y_train)
ablation_mae = mean_absolute_error(y_test, ablation_model.predict(X_test[FEATURES_NO_CLUSTER]))
ablation_r2  = r2_score(y_test, ablation_model.predict(X_test[FEATURES_NO_CLUSTER]))

full_mae = results[best_tuned_name]['test_mae']
full_r2  = results[best_tuned_name]['test_r2']

print(f'   Without cluster features → MAE: {ablation_mae:.4f}  R²: {ablation_r2:.4f}')
print(f'   With cluster features    → MAE: {full_mae:.4f}  R²: {full_r2:.4f}')
improvement = (ablation_mae - full_mae) / ablation_mae * 100
print(f'   MAE improvement from clusters: {improvement:.1f}%')
print(f'   → Cluster features {"HELP" if improvement > 0 else "DO NOT HELP"} the model')

## Step 12 — Predicted vs Actual

In [ ]:
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.4, s=20, color='#e10600', edgecolors='none')
lims = [min(y_test.min(), y_pred_best.min()) - 1, max(y_test.max(), y_pred_best.max()) + 1]
axes[0].plot(lims, lims, 'k--', linewidth=1, alpha=0.6, label='Perfect prediction')
axes[0].set_xlabel('Actual Position Gain'); axes[0].set_ylabel('Predicted Position Gain')
axes[0].set_title(f'{best_tuned_name}\nPredicted vs Actual (2025 Test Set)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=40, color='#1f77b4', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean={residuals.mean():.2f}')
axes[1].set_xlabel('Residual (Actual − Predicted)'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'predictions_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 13 — SHAP Values

In [ ]:
print('🔄 Computing SHAP values...')
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
print(f'✅ SHAP values computed: {shap_values.shape}')

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False, max_display=13)
plt.title('SHAP Summary — Feature Impact on Position Gain', fontweight='bold', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/shap_summary.png')

In [ ]:
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, plot_type='bar', show=False, max_display=13)
plt.title('Mean |SHAP Value| — Global Feature Importance', fontweight='bold', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_bar.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/shap_bar.png')

## Step 14 — Save Models

In [ ]:
models_to_save = {
    'rf_tuned.pkl':  rf_best,
    'xgb_tuned.pkl': xgb_best,
    'gb_tuned.pkl':  gb_best,
}

for filename, model in models_to_save.items():
    path = os.path.join(MODELS_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'💾 Saved: models/{filename}')

print('\n✅ All models saved.')
print(f'\n🏁 FINAL RESULT: {best_tuned_name}')
print(f'   Test MAE:  {results[best_tuned_name]["test_mae"]:.4f}')
print(f'   Test RMSE: {results[best_tuned_name]["test_rmse"]:.4f}')
print(f'   Test R²:   {results[best_tuned_name]["test_r2"]:.4f}')

mean_mae = results['Baseline: Mean Predictor']['test_mae']
print(f'   Improvement vs mean baseline: {(mean_mae - results[best_tuned_name]["test_mae"])/mean_mae*100:.1f}%')